# Text Embedding

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/blob/main/language/getting-started/intro_palm_api.ipynb">
      <img src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
</table>


## Getting Started

### Install Vertex AI SDK

In [1]:
!pip install google-cloud-aiplatform --upgrade --user

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 31.3 MB/s eta 0:00:0000:0100:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Use the button to restart the kernel. For Vertex AI Workbench you can restart the terminal using the button on top.

In [1]:
# Restart

- If you are running this notebook in a local development environment:
  - Install the [Google Cloud SDK](https://cloud.google.com/sdk).
  - Obtain authentication credentials. Create local credentials by running the following command and following the oauth2 flow (read more about the command [here](https://cloud.google.com/sdk/gcloud/reference/beta/auth/application-default/login)):

    ```bash
    gcloud auth application-default login
    ```

### Available models

*   `textembedding-gecko@001` : Returns model embeddings for text inputs.

ref: [foundational models in the Generative AI Studio documentation](https://cloud.google.com/vertex-ai/docs/generative-ai/learn/models#foundation_models).


### Import libraries

In [56]:
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.metrics.pairwise import cosine_similarity
from vertexai.language_models import (
    TextEmbeddingModel,
    TextEmbeddingInput,
    InputOutputTextPair,
    CodeGenerationModel
)

## Embedding model with `textembedding-gecko@001`

Text embeddings are a dense, often low-dimensional, vector representation of a piece of content such that, if two pieces of content are semantically similar, their respective embeddings are located near each other in the embedding vector space. This representation can be used to solve common NLP tasks, such as:

* **Semantic search**: Search text ranked by semantic similarity.
* **Recommendation**: Return items with text attributes similar to the given text.
* **Classification**: Return the class of items whose text attributes are similar to the given text.
* **Clustering**: Cluster items whose text attributes are similar to the given text.
* **Outlier Detection**: Return items where text attributes are least related to the given text.

Please refer to the [text embedding model documentation](https://cloud.google.com/vertex-ai/docs/generative-ai/embeddings/get-text-embeddings) for more information.

In [57]:
#embedding_model = TextEmbeddingModel.from_pretrained("textembedding-gecko@001")
embedding_model = TextEmbeddingModel.from_pretrained("textembedding-gecko-multilingual")

In [58]:
content = """
W style 品牌介紹

品牌定位：

    專注於女性時尚服飾和配件
    提供高品質、時尚且實穿的服飾
    迎合現代女性的品味和生活方式

產品線：

    **服飾：**連衣裙、上衣、褲子、裙子、外套
    **配件：**包包、鞋子、首飾、帽子、圍巾
    **居家服：**睡衣、家居服、拖鞋

設計風格：

    **簡約優雅：**注重線條簡潔、剪裁合身
    **時尚百搭：**易於搭配，適合各種場合
    **精緻細節：**注重面料質感、刺繡和印花等細節

目標客群：

    25-45 歲的現代女性
    注重時尚、品質和實用性
    追求優雅、自信和獨立的風格

品牌價值：

    **品質保證：**嚴格把關面料和製作工藝
    **時尚品味：**由專業設計師團隊打造
    **顧客至上：**提供優質的售後服務和購物體驗

品牌特色：

    **線上購物平台：**提供便利的線上購物體驗
    **實體門市：**台北、台中、高雄等主要城市設有門市
    **會員制度：**提供會員專屬優惠和活動
    **時尚部落格：**分享時尚趨勢、穿搭靈感和品牌故事
    **社群媒體：**積極經營社群媒體，與顧客互動並展示最新產品

品牌故事：

W style 由一群熱愛時尚的女性創立，她們希望打造一個讓女性展現自信和優雅的品牌。品牌名稱「W」代表「Women」，象徵著品牌對女性的重視和支持。

W style 致力於提供高品質、時尚且實穿的服飾，讓女性在各種場合都能展現自己的獨特風格。品牌不斷創新，推出符合現代女性需求的產品，並堅持以顧客為中心的理念，提供優質的購物體驗。
"""

#### Embeddings and Pandas DataFrames

If your text is stored in a column of a DataFrame, you can create a new column with the embeddings with the example below.

In [59]:
text = [
    content
]

df = pd.DataFrame(text, columns=["text"])
df

,text
0,\nW style 品牌介紹\n\n品牌定位：\n\n 專注於女性時尚服飾和配件\n ...


Create a new column, `embeddings`, using the [apply](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html) function in pandas with the embeddings model.

In [66]:
df["obj_embedding"] = df.apply(
    lambda x: embedding_model.get_embeddings([TextEmbeddingInput(text=x.text, task_type="RETRIEVAL_DOCUMENT")])[0], axis=1
)
df["embeddings"] = df["obj_embedding"].apply(lambda x: x.values)
df["statistics"] = df["obj_embedding"].apply(lambda x: x.statistics)
df = df.drop("obj_embedding", axis=1)

df

,text,embeddings,statistics
0,\nW style 品牌介紹\n\n品牌定位：\n\n 專注於女性時尚服飾和配件\n ...,"[0.03675073757767677, -0.003267692169174552, 0...","TextEmbeddingStatistics(token_count=452.0, tru..."


#### Comparing similarity of text examples using cosine similarity

By converting text into embeddings, you can compute similarity scores. There are many ways to compute similarity scores, and one common technique is using [cosine similarity](https://en.wikipedia.org/wiki/Cosine_similarity).

In the example from above, two of the sentences in the `text` column relate to enjoying a _movie_, and the other two relates to enjoying _coding_. Cosine similarity scores should be higher (closer to 1.0) when doing pairwise comparisons between semantically-related sentences, and scores should be lower between semantically-different sentences.

The DataFrame output below shows the resulting cosine similarity scores between the embeddings:

In [61]:
cos_sim_array = cosine_similarity(list(df.embeddings.values))

# display as DataFrame
df_cos = pd.DataFrame(cos_sim_array, index=text, columns=text)
df_cos

,\nW style 品牌介紹\n\n品牌定位：\n\n 專注於女性時尚服飾和配件\n 提供高品質、時尚且實穿的服飾\n 迎合現代女性的品味和生活方式\n\n產品線：\n\n **服飾：**連衣裙、上衣、褲子、裙子、外套\n **配件：**包包、鞋子、首飾、帽子、圍巾\n **居家服：**睡衣、家居服、拖鞋\n\n設計風格：\n\n **簡約優雅：**注重線條簡潔、剪裁合身\n **時尚百搭：**易於搭配，適合各種場合\n **精緻細節：**注重面料質感、刺繡和印花等細節\n\n目標客群：\n\n 25-45 歲的現代女性\n 注重時尚、品質和實用性\n 追求優雅、自信和獨立的風格\n\n品牌價值：\n\n **品質保證：**嚴格把關面料和製作工藝\n **時尚品味：**由專業設計師團隊打造\n **顧客至上：**提供優質的售後服務和購物體驗\n\n品牌特色：\n\n **線上購物平台：**提供便利的線上購物體驗\n **實體門市：**台北、台中、高雄等主要城市設有門市\n **會員制度：**提供會員專屬優惠和活動\n **時尚部落格：**分享時尚趨勢、穿搭靈感和品牌故事\n **社群媒體：**積極經營社群媒體，與顧客互動並展示最新產品\n\n品牌故事：\n\nW style 由一群熱愛時尚的女性創立，她們希望打造一個讓女性展現自信和優雅的品牌。品牌名稱「W」代表「Women」，象徵著品牌對女性的重視和支持。\n\nW style 致力於提供高品質、時尚且實穿的服飾，讓女性在各種場合都能展現自己的獨特風格。品牌不斷創新，推出符合現代女性需求的產品，並堅持以顧客為中心的理念，提供優質的購物體驗。\n
\nW style 品牌介紹\n\n品牌定位：\n\n 專注於女性時尚服飾和配件\n 提供高品質、時尚且實穿的服飾\n 迎合現代女性的品味和生活方式\n\n產品線：\n\n **服飾：**連衣裙、上衣、褲子、裙子、外套\n **配件：**包包、鞋子、首飾、帽子、圍巾\n **居家服：**睡衣、家居服、拖鞋\n\n設計風格：\n\n **簡約優雅：**注重線條簡潔、剪裁合身\n **時尚百搭：**易於搭配，適合各種場合\n **精緻細節：**注重面料質感、刺繡和印花等細節\n\n目標客群：\n\n 25-45 歲的現代女性\n 注重時尚、品質和實用性\n 追求優雅、自信和獨立的風格\n\n品牌價值：\n\n **品質保證：**嚴格把關面料和製作工藝\n **時尚品味：**由專業設計師團隊打造\n **顧客至上：**提供優質的售後服務和購物體驗\n\n品牌特色：\n\n **線上購物平台：**提供便利的線上購物體驗\n **實體門市：**台北、台中、高雄等主要城市設有門市\n **會員制度：**提供會員專屬優惠和活動\n **時尚部落格：**分享時尚趨勢、穿搭靈感和品牌故事\n **社群媒體：**積極經營社群媒體，與顧客互動並展示最新產品\n\n品牌故事：\n\nW style 由一群熱愛時尚的女性創立，她們希望打造一個讓女性展現自信和優雅的品牌。品牌名稱「W」代表「Women」，象徵著品牌對女性的重視和支持。\n\nW style 致力於提供高品質、時尚且實穿的服飾，讓女性在各種場合都能展現自己的獨特風格。品牌不斷創新，推出符合現代女性需求的產品，並堅持以顧客為中心的理念，提供優質的購物體驗。\n,1.0


To make this easier to understand, you can use a heatmap. Naturally, text is most similar when they are identical (score of 1.0). The next highest scores are when sentences are semantically similar. The lowest scores are when sentences are quite different in meaning.

In [62]:
#ax = sns.heatmap(df, annot=True, cmap="crest")
#ax.xaxis.tick_top()
#ax.set_xticklabels(text, rotation=90)

In [63]:
vec_0 = df['embeddings'][0]

In [64]:
len(vec_0)

768

In [67]:
df['statistics'][0]

TextEmbeddingStatistics(token_count=452.0, truncated=False)

In [68]:
str(['%1.2e' % (ival) for ival in vec_0])

"['3.68e-02', '-3.12e-03', '3.58e-02', '1.70e-02', '-4.73e-02', '-4.59e-02', '-4.79e-02', '1.62e-02', '4.05e-02', '2.44e-02', '1.29e-02', '3.43e-02', '3.20e-02', '-1.34e-02', '1.50e-02', '5.76e-02', '3.91e-02', '-4.90e-02', '-3.64e-02', '-6.08e-03', '-8.16e-03', '3.37e-02', '-2.34e-02', '-1.13e-02', '4.39e-02', '1.01e-02', '2.49e-02', '3.59e-02', '-4.72e-02', '6.14e-02', '-2.33e-02', '-2.32e-02', '-1.19e-02', '4.22e-02', '-9.98e-03', '-1.83e-02', '6.73e-03', '-1.56e-02', '1.38e-02', '2.45e-02', '6.26e-02', '-8.04e-02', '-4.82e-02', '2.12e-02', '1.46e-02', '2.09e-03', '2.16e-02', '-7.41e-03', '7.39e-02', '2.30e-02', '3.35e-02', '-4.80e-02', '-1.42e-02', '-2.36e-02', '-6.02e-02', '-4.61e-02', '-5.67e-02', '1.22e-02', '-2.24e-02', '4.97e-02', '2.43e-02', '1.80e-02', '-3.37e-02', '5.51e-02', '5.62e-02', '-2.66e-02', '-5.28e-02', '3.23e-02', '3.97e-02', '-2.21e-04', '-2.89e-02', '3.93e-03', '-4.86e-02', '-1.80e-02', '1.85e-03', '4.44e-02', '2.16e-02', '-6.61e-02', '-2.49e-03', '-5.78e-02', 